# FST master analysis

This notebook is organized into two independent stages:

1. **Preprocessing**: convert raw EthoVision activity files into untransformed first-2-minute and last-4-minute phenotype tables, merge experimental metadata, and export those tables.
2. **Statistical analysis**: reload the exported untransformed phenotype tables and perform transformations, modelling, model comparisons, and figure generation.

Users who already have the processed phenotype tables can begin directly at **Step 2**.


# Step 1 — Preprocessing

Convert the raw activity files into FST immobility phenotypes and attach the experimental metadata required for downstream analysis.


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import scipy.stats as stats
from scipy.stats import mannwhitneyu
from scipy.stats import shapiro, kstest
from scipy.stats import levene
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from pathlib import Path
import re
from scipy.stats import boxcox
import os

### Define thresholds and preprocessing functions

Define the frame duration, strain-specific activity thresholds, and helper functions used to read and process the activity traces.


In [ ]:
FRAME_DURATION_SECONDS = 0.04

DEFAULT_THRESHOLDS = {
    'MRL': 2.2799,
    'MRL_2': 2.721256,
    'DBA': 0.5883,
    'CBA': 1.0297,
    'B6': 1.3238
}


def read_zoom_file(zoom_file):
    zoom_df = pd.read_excel(
        zoom_file,
        usecols=["Experiment", "Pixel Distance"],
        engine="openpyxl"
    )

    zoom_df["Experiment"] = (
        zoom_df["Experiment"]
        .astype(str)
        .str.extract(r"(\d+)")
        .astype(int)
    )

    first_px = zoom_df["Pixel Distance"].iat[0]
    zoom_df["Zoom"] = (zoom_df["Pixel Distance"] / first_px) ** 2

    return zoom_df


def read_excel_activity_file(xl_path):
    xl_path = Path(xl_path)

    header = pd.read_excel(
        xl_path,
        header=None,
        usecols="B",
        nrows=11,
        engine="openpyxl"
    )

    raw_exp = str(header.iloc[9, 0])
    strain = str(header.iloc[10, 0]).strip()

    exp_number = int(re.search(r"(\d+)", raw_exp).group(1))

    activity = pd.read_excel(
        xl_path,
        header=None,
        usecols="N",
        skiprows=37,
        nrows=9000,
        engine="openpyxl"
    ).iloc[:, 0]

    activity = (
        activity
        .replace("-", np.nan)
        .pipe(pd.to_numeric, errors="coerce")
    )

    return exp_number, strain, activity


def get_zoom_value(exp_number, zoom_df):
    matches = zoom_df.loc[zoom_df["Experiment"] == exp_number, "Zoom"]

    if len(matches) == 0:
        raise KeyError(f"No zoom value found for experiment {exp_number}")

    if len(matches) > 1:
        raise ValueError(f"Multiple zoom values found for experiment {exp_number}")

    return float(matches.iloc[0])


def process_activity_files(
    folder,
    zoom_file,
    thresholds=DEFAULT_THRESHOLDS,
    pattern="*.xlsx",
):
    """Process EthoVision activity files and return FST immobility summaries.

    Each source workbook is read once, corrected for zoom, smoothed with the
    original 100-frame rolling mean, and converted to mobility/immobility calls
    using the strain-specific thresholds above. No intermediate cache files are
    written to disk.
    """
    folder = Path(folder)
    zoom_df = read_zoom_file(zoom_file)

    rows_first = []
    rows_rest = []
    meta_rows = []
    error_rows = []

    activity_files = sorted(folder.glob(pattern))
    total_files = len(activity_files)

    for i, xl_path in enumerate(activity_files, start=1):
        try:
            exp_number, strain, activity = read_excel_activity_file(xl_path)
            zoom_value = get_zoom_value(exp_number, zoom_df)

            corrected_activity = activity / zoom_value
            smoothed_activity = (
                corrected_activity
                .rolling(window=100, center=True, min_periods=1)
                .mean()
            )

            threshold_key = strain
            if strain == "MRL":
                threshold_key = "MRL" if exp_number <= 199 else "MRL_2"

            if threshold_key not in thresholds:
                raise KeyError(f"No threshold defined for strain {threshold_key}")

            threshold = thresholds[threshold_key]

            first_part = smoothed_activity.iloc[:3000]
            rest_part = smoothed_activity.iloc[3000:]

            first_calls = np.where(first_part >= threshold, "m", "i")
            rest_calls = np.where(rest_part >= threshold, "m", "i")

            first_i = np.sum(first_calls == "i")
            rest_i = np.sum(rest_calls == "i")

            rows_first.append({
                "ID": exp_number,
                "Strain": strain,
                "threshold_used": threshold_key,
                "segment": "first_3000",
                "immobile_frames": first_i,
                "immobile_time": first_i * FRAME_DURATION_SECONDS,
            })

            rows_rest.append({
                "ID": exp_number,
                "Strain": strain,
                "threshold_used": threshold_key,
                "segment": "last_6000",
                "immobile_frames": rest_i,
                "immobile_time": rest_i * FRAME_DURATION_SECONDS,
            })

            meta_rows.append({
                "Experiment number": exp_number,
                "Strain": strain,
                "source_file": xl_path.name,
                "status": "processed",
            })

        except Exception as exc:
            error_rows.append({
                "source_file": xl_path.name,
                "error": repr(exc),
            })

        if i % 10 == 0 or i == total_files:
            pct = (i / total_files) * 100
            print(f"Processed {i}/{total_files} files ({pct:.1f}%)")

    df_first = pd.DataFrame(rows_first).sort_values("ID").reset_index(drop=True)
    df_rest = pd.DataFrame(rows_rest).sort_values("ID").reset_index(drop=True)
    metadata_df = pd.DataFrame(meta_rows).sort_values("Experiment number").reset_index(drop=True)
    errors_df = pd.DataFrame(error_rows)

    return df_first, df_rest, metadata_df, errors_df


### Set input and output paths

The example below assumes the repository contains a `data/` directory for input files and a `results/` directory for generated outputs. Adjust these paths if your local folder structure differs.


In [ ]:
# Run the notebook from the repository root, or edit PROJECT_ROOT below.
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

# Raw/input files and directories
RAW_ACTIVITY_DIR = DATA_DIR / "ethovision_activity"
ZOOM_FILE = DATA_DIR / "FST_runs.xlsx"
EXPERIMENTAL_SHEET_FILE = DATA_DIR / "experimental_sheet.csv"
GENOTYPES_FILE = DATA_DIR / "genotypes.xlsx"

# Files produced by Step 1 and used as the starting point for Step 2
FIRST_UNTRANSFORMED_FILE = PROCESSED_DATA_DIR / "fst_first_2min_untransformed.csv"
REST_UNTRANSFORMED_FILE = PROCESSED_DATA_DIR / "fst_last_4min_untransformed.csv"

# Create output directories if they do not already exist.
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


### Process activity files and calculate immobility

Read each source activity workbook once, apply the zoom correction and smoothing step, and calculate immobility separately for the first 2 minutes and remaining 4 minutes of the FST. This step intentionally does not create intermediate cache files.


In [ ]:
df_first, df_rest, metadata_df, errors_df = process_activity_files(
    folder=RAW_ACTIVITY_DIR,
    zoom_file=ZOOM_FILE,
    thresholds=DEFAULT_THRESHOLDS,
)

if not errors_df.empty:
    print(f"{len(errors_df)} file(s) could not be processed:")
    display(errors_df)


The resulting `df_first` and `df_rest` data frames are used by the downstream analysis cells. `metadata_df` records the successfully processed source files, while `errors_df` records files that could not be processed.


## 2. Add genotype and experimental metadata


### Load colony-sheet metadata

Load genotype, sex, strain, cage, and related experimental information and merge these variables with the FST phenotype tables.


In [ ]:
experimental_sheet=pd.read_csv(EXPERIMENTAL_SHEET_FILE, usecols=['Cage', 'Platform ID', 'Our ID', 'Sex', 'Weight'])
# Edit 2
exp_index = experimental_sheet[experimental_sheet['Our ID'] == 'E110'].index[0]
experimental_sheet_one = experimental_sheet.loc[:exp_index-1]
#Edit 3
experimental_sheet_two = experimental_sheet_one.dropna(how='any')
experimental_sheet_three = experimental_sheet_two[experimental_sheet_two['Platform ID'].astype(str).str.startswith('2')]

genotypes = pd.read_excel(GENOTYPES_FILE)
lookup_dict = genotypes.set_index('Our ID')['Genotype'].to_dict()

# Step 2: Use map() to create a new column in df1
experimental_sheet_three['Genotype'] = experimental_sheet_three['Our ID'].map(lookup_dict)
checks1 = experimental_sheet_three[experimental_sheet_three['Genotype'].isna() & experimental_sheet_three['Our ID'].astype(str).str.startswith('E')].index
#print(checks1)

rows_to_drop = list(checks1) + [i + 1 for i in checks1 if i + 1 < len(experimental_sheet_three)]

# Step 3: Drop those rows from the original DataFrame
experimental_cleaned = experimental_sheet_three.drop(rows_to_drop).reset_index(drop=True)
replaced_count = 0

# Loop through every second row starting at index 1
for i in range(1, len(experimental_cleaned), 2):
    if pd.isna(experimental_cleaned.at[i, 'Genotype']):
        experimental_cleaned.at[i, 'Genotype'] = experimental_cleaned.at[i - 1, 'Genotype']
        replaced_count += 1

print(f"Number of NaNs replaced from the previous row: {replaced_count}")
nan_percent = experimental_cleaned['Genotype'].isna().mean() * 100
print(f"Percentage of NaNs in 'your_column': {nan_percent:.2f}%")
experimental_cleaned["Cage"] = (
    experimental_cleaned["Cage"]
        .astype(str)                # ensure string dtype
        .str.extract(r"(\d+)", expand=False)  # pull the digit run
        .astype(int)                # convert to integers
)
mask = experimental_cleaned['Our ID'].str.startswith(tuple("E"), na=False)   # True for rows to drop
exps = experimental_cleaned.loc[~mask]

In [6]:
df_first["ID"] = df_first["ID"].astype(int)



# 2️⃣  Perform a left-join and REASSIGN to df1  (this applies the merge)
df_first = (
    df_first
      .merge(exps, left_on="ID", right_on="Cage", how="left")
      .drop(columns="Cage")          # optional: remove the duplicate key
)
df_first["ID"] = df_first["ID"].astype(int)



df_rest["ID"] = df_rest["ID"].astype(int)



# 2️⃣  Perform a left-join and REASSIGN to df1  (this applies the merge)
df_rest = (
    df_rest
      .merge(exps, left_on="ID", right_on="Cage", how="left")
      .drop(columns="Cage")          # optional: remove the duplicate key
)
df_rest["ID"] = df_rest["ID"].astype(int)

### Export untransformed phenotype tables

Save the completed, **untransformed** phenotype tables after metadata have been merged. These two CSV files are the hand-off between preprocessing and statistical analysis and can also be archived as analysis-ready data.


In [ ]:
df_first.to_csv(FIRST_UNTRANSFORMED_FILE, index=False)
df_rest.to_csv(REST_UNTRANSFORMED_FILE, index=False)

print(f"Saved first-2-minute data to: {FIRST_UNTRANSFORMED_FILE}")
print(f"Saved last-4-minute data to: {REST_UNTRANSFORMED_FILE}")


# Step 2 — Statistical analysis

This section can be run independently of Step 1. It begins by loading the untransformed phenotype tables produced during preprocessing. If these CSV files are already available, users can start here without reprocessing the raw EthoVision files.


## Load untransformed FST phenotype tables


In [ ]:
# Step 2 can be started from a fresh kernel.
import pandas as pd
import numpy as np
import seaborn as sns
import scipy.stats as stats
from scipy.stats import mannwhitneyu, shapiro, kstest, levene, ttest_ind, boxcox
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from pathlib import Path
import re
import os

# Run from the repository root, or edit PROJECT_ROOT as needed.
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

FIRST_UNTRANSFORMED_FILE = PROCESSED_DATA_DIR / "fst_first_2min_untransformed.csv"
REST_UNTRANSFORMED_FILE = PROCESSED_DATA_DIR / "fst_last_4min_untransformed.csv"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)


## Box–Cox transformation


### Transform the immobility phenotype

Apply the Box–Cox transformation used for the downstream statistical analyses and retain the estimated transformation parameters.


In [8]:
def boxcox_transform_df(df):
    # Columns to exclude
    exclude_cols = list(df.columns[:2]) + list(df.columns[13:18])

    # Columns to transform
    to_transform_cols = ['immobile_time']
    # Split
    untouched_df = df[exclude_cols].copy()
    to_transform_df = df[to_transform_cols].copy().astype(float)

    # Clean: Replace NaNs and 0s with 0.01
    to_transform_df = to_transform_df.fillna(1)
    to_transform_df[to_transform_df == 0] = 1

    transformed_data = {}
    lambdas = {}

    for col in to_transform_df.columns:
        transformed, lmbda = boxcox(to_transform_df[col])
        transformed_data[col] = transformed
        lambdas[col] = lmbda

    transformed_df = pd.DataFrame(transformed_data, index=df.index)

    # Reconstruct in original column order
    final_df = df.copy()
    final_df[to_transform_cols] = transformed_df

    return final_df, lambdas

In [9]:
first_transformed_df, first_lambdas = boxcox_transform_df(df_first)
rest_transformed_df, rest_lambdas = boxcox_transform_df(df_rest)

### Per strain analysis 


In [83]:
MRL_first = first_transformed_df[first_transformed_df['Strain'] == 'MRL']
DBA_first = first_transformed_df[first_transformed_df['Strain'] == 'DBA']
CBA_first = first_transformed_df[first_transformed_df['Strain'] == 'CBA']
B6_first = first_transformed_df[first_transformed_df['Strain'] == 'B6']

MRL_rest = rest_transformed_df[rest_transformed_df['Strain'] == 'MRL']
DBA_rest = rest_transformed_df[rest_transformed_df['Strain'] == 'DBA']
CBA_rest = rest_transformed_df[rest_transformed_df['Strain'] == 'CBA']
B6_rest = rest_transformed_df[rest_transformed_df['Strain'] == 'B6']

### First 2 minutes: strain-specific models

Fit strain-specific models to evaluate additive and interaction effects of sex and cage-mate genotype during the first 2 minutes.


In [3]:
import statsmodels.api as sm
model_data = MRL_first.copy()

m0 = smf.ols("immobile_time ~ 1", data=model_data).fit()

m1 = smf.ols("immobile_time ~ Sex", data=model_data).fit()

m2 = smf.ols("immobile_time ~ Sex + Genotype", data=model_data).fit()

m3 = smf.ols("immobile_time ~ Sex * Genotype", data=model_data).fit()

m4 = smf.ols("immobile_time ~ Genotype", data=model_data).fit()

aic_values = {
    "m0 (null)": m0.aic,
    "m1 (Sex)": m1.aic,
    "m2 (Sex + Genotype)": m2.aic,
    "m3 (Sex * Genotype)": m3.aic,
    "m4 (Genotype)": m4.aic
}

for name, val in aic_values.items():
    print(f"{name}: {val:.2f}")

aic_df = pd.DataFrame({
"Model": list(aic_values.keys()),
"AIC": list(aic_values.values())
})

aic_df["ΔAIC"] = aic_df["AIC"] - aic_df["AIC"].min()
print(aic_df)

anova_res2 = sm.stats.anova_lm(m1, m3)
print(anova_res2)


NameError: name 'MRL_first' is not defined

### Last 4 minutes: strain-specific models

Repeat the strain-specific model analysis for the last 4 minutes.


In [93]:
import statsmodels.api as sm
model_data = MRL_rest.copy()

m0 = smf.ols("immobile_time ~ 1", data=model_data).fit()

m1 = smf.ols("immobile_time ~ Sex", data=model_data).fit()

m2 = smf.ols("immobile_time ~ Sex + Genotype", data=model_data).fit()

m3 = smf.ols("immobile_time ~ Sex * Genotype", data=model_data).fit()

m4 = smf.ols("immobile_time ~ Genotype", data=model_data).fit()

aic_values = {
    "m0 (null)": m0.aic,
    "m1 (Sex)": m1.aic,
    "m2 (Sex + Genotype)": m2.aic,
    "m3 (Sex * Genotype)": m3.aic,
    "m4 (Genotype)": m4.aic
}

for name, val in aic_values.items():
    print(f"{name}: {val:.2f}")

aic_df = pd.DataFrame({
"Model": list(aic_values.keys()),
"AIC": list(aic_values.values())
})

aic_df["ΔAIC"] = aic_df["AIC"] - aic_df["AIC"].min()
print(aic_df)


anova_res2 = sm.stats.anova_lm(m1, m3)
print(anova_res2)


m0 (null): 4011.81
m1 (Sex): 3922.35
m2 (Sex + Genotype): 3924.31
m3 (Sex * Genotype): 3926.29
m4 (Genotype): 3929.40
                 Model          AIC       ΔAIC
0            m0 (null)  4011.809842  89.456916
1             m1 (Sex)  3922.352926   0.000000
2  m2 (Sex + Genotype)  3924.309955   1.957029
3  m3 (Sex * Genotype)  3926.290860   3.937933
4        m4 (Genotype)  3929.401094   7.048167
   df_resid           ssr  df_diff       ss_diff         F    Pr(>F)
0      89.0  2.671631e+19      0.0           NaN       NaN       NaN
1      88.0  2.670369e+19      1.0  1.261270e+16  0.041564  0.838923
   df_resid           ssr  df_diff       ss_diff         F    Pr(>F)
0      89.0  2.671631e+19      0.0           NaN       NaN       NaN
1      87.0  2.669809e+19      2.0  1.821574e+16  0.029679  0.970766
   df_resid           ssr  df_diff       ss_diff         F    Pr(>F)
0      92.0  2.891688e+19      0.0           NaN       NaN       NaN
1      89.0  2.886779e+19      3.0  4.909072e+16

"\nwith pd.ExcelWriter('/Users/pmatani/OneDrive - CRG - Centre de Regulacio Genomica/Documents - Baud lab/Pavans_projects/IGE Paper/New_figures_13_04/AIC_table_FST4mins_Sex_B6_newmethod.xlsx') as writer:\n    aic_df.to_excel(writer, sheet_name='AIC', index=False)\n\n    # Write first ANOVA with label\n    label1 = pd.DataFrame([['Additive Effect']])\n    label1.to_excel(writer, sheet_name='ANOVA', index=False, header=False, startrow=0)\n    anova_res1.to_excel(writer, sheet_name='ANOVA', startrow=1)\n\n    # Write second ANOVA below, with a gap and label\n    startrow = len(anova_res1) + 4  # +4 leaves a blank row gap\n    label2 = pd.DataFrame([['Interaction Effect']])\n    label2.to_excel(writer, sheet_name='ANOVA', index=False, header=False, startrow=startrow)\n    anova_res2.to_excel(writer, sheet_name='ANOVA', startrow=startrow + 1)\n"

In [ ]:
import pandas as pd

# Results generated by this notebook are written to RESULTS_DIR.
base_path = RESULTS_DIR

strains = ['MRL', 'DBA', 'CBA', 'B6']
files = {strain: base_path / f'AIC_table_FST2mins_Sex_{strain}_newmethod.xlsx' for strain in strains}

def extract_anova_stats(filepath, strain):
    df = pd.read_excel(filepath, sheet_name='ANOVA', header=None)
    
    # Find rows where labels 'Additive Effect' and 'Interaction Effect' are
    label_rows = df[df[0].astype(str).str.contains('Additive Effect|Interaction Effect', na=False)].index.tolist()
    
    results = {}
    for i, label in zip(label_rows, ['Additive', 'Interaction']):
        # Header is the row after the label, data starts after that
        header_row = i + 1
        data_start = i + 2
        
        # Find the end of this block (next label or end of file)
        next_label = label_rows[label_rows.index(i) + 1] if i != label_rows[-1] else len(df)
        
        block = df.iloc[data_start:next_label].copy()
        block.columns = df.iloc[header_row].values
        
        
        # Keep only F and p-value columns — adjust col names if needed
        f_col = [c for c in block.columns if 'F' in str(c)][0]
        p_col = [c for c in block.columns if 'PR' in str(c).upper() or 'p-value' in str(c).lower() or 'pvalue' in str(c).lower()][0]
        block = block.dropna(subset=[f_col, p_col])  # now safe to use
        block = block[['index' if 'index' in block.columns else block.columns[0], f_col, p_col]].copy()
        block.columns = ['Term', f'F_{strain}', f'pval_{strain}']
        block['Effect'] = label
        block = block.set_index(['Effect', 'Term'])
        results[label] = block
    
    return pd.concat(results.values())

# Extract from all strains
all_dfs = [extract_anova_stats(files[strain], strain) for strain in strains]

# Combine side by side
combined = pd.concat(all_dfs, axis=1)
combined = combined.reset_index()

# Save
output_path = base_path / 'Combined_ANOVA_allstrains2mins.xlsx'
with pd.ExcelWriter(output_path) as writer:
    combined.to_excel(writer, sheet_name='Combined_ANOVA', index=False)

print("Done! Saved to:", output_path)
print(combined)

## Fighting analysis with Social epistasis

In [13]:
model_data = first_transformed_df.copy()

# ============================================================
# Add Background column
# ============================================================

model_data["Background"] = np.where(
    model_data["Strain"] == "B6",
    "Same",
    "Mixed"
)

# Optional: make ordering explicit
model_data["Background"] = pd.Categorical(
    model_data["Background"],
    categories=["Same", "Mixed"]
)

print(model_data[["Strain", "Background"]].value_counts())


# ============================================================
# Original OLS models using Strain
# ============================================================

m0 = smf.ols(
    "immobile_time ~ 1",
    data=model_data
).fit()

m1 = smf.ols(
    "immobile_time ~ Strain",
    data=model_data
).fit()

m2 = smf.ols(
    "immobile_time ~ Sex",
    data=model_data
).fit()

m3 = smf.ols(
    "immobile_time ~ Strain + Sex",
    data=model_data
).fit()

m4 = smf.ols(
    "immobile_time ~ Strain * Sex",
    data=model_data
).fit()

m5 = smf.ols(
    "immobile_time ~ Strain + Sex + Genotype",
    data=model_data
).fit()

m6 = smf.ols(
    "immobile_time ~ Strain + Sex * Genotype",
    data=model_data
).fit()

m7 = smf.ols(
    "immobile_time ~ Strain * Genotype + Sex",
    data=model_data
).fit()

m8 = smf.ols(
    "immobile_time ~ Strain * Sex + Genotype",
    data=model_data
).fit()

m9 = smf.ols(
    "immobile_time ~ Strain * Sex * Genotype",
    data=model_data
).fit()


# ============================================================
# Parallel models using Background instead of Strain
# ============================================================

# Equivalent of m1
m10 = smf.ols(
    "immobile_time ~ Strain +  Sex + Genotype + Background:Genotype",
    data=model_data
).fit()

# Equivalent of m3
m11 = smf.ols(
    "immobile_time ~ Strain + Sex + Genotype + Strain:Sex + Sex:Genotype + Background:Genotype + Sex:Background:Genotype",
    data=model_data
).fit()

# ============================================================
# AIC comparison
# ============================================================

aic_values = {

    # Shared models
    "m0 (null)": m0.aic,
    "m2 (Sex)": m2.aic,

    # Strain models
    "m1 (Strain)": m1.aic,
    "m3 (Strain + Sex)": m3.aic,
    "m4 (Strain * Sex)": m4.aic,
    "m5 (Strain + Sex + Genotype)": m5.aic,
    "m6 (Strain + Sex * Genotype)": m6.aic,
    "m7 (Strain * Genotype + Sex)": m7.aic,
    "m8 (Strain * Sex + Genotype)": m8.aic,
    "m9 (Strain * Sex * Genotype)": m9.aic,

    # Background models
    "m10 (Strain + Sex + Genotype + SameBackground:Genotype)": m10.aic,
    "m11 (Strain + Sex + Genotype + Strain:Sex + Sex:Genotype + SameBackground:Genotype + Sex:SameBackground:Genotype)": m11.aic
}


# ============================================================
# Make AIC dataframe
# ============================================================

aic_df = pd.DataFrame({
    "Model": list(aic_values.keys()),
    "AIC": list(aic_values.values())
})

aic_df["ΔAIC"] = aic_df["AIC"] - aic_df["AIC"].min()
"""
# Sort best model first
aic_df = aic_df.sort_values(
    "AIC",
    ascending=True
).reset_index(drop=True)
"""
print(aic_df)
aic_df.to_excel(RESULTS_DIR / 'AIC_table_FST_2mins_Sex_Strain_Genotype_newmethod_fighting.xlsx', index=False)

Strain  Background
B6      Same          106
DBA     Mixed          99
CBA     Mixed          97
MRL     Mixed          93
Name: count, dtype: int64
                                                Model          AIC        ΔAIC
0                                           m0 (null)  2924.864736  207.849026
1                                            m2 (Sex)  2865.047450  148.031741
2                                         m1 (Strain)  2772.934449   55.918740
3                                   m3 (Strain + Sex)  2717.501344    0.485634
4                                   m4 (Strain * Sex)  2719.735322    2.719612
5                        m5 (Strain + Sex + Genotype)  2717.015709    0.000000
6                        m6 (Strain + Sex * Genotype)  2719.014277    1.998568
7                        m7 (Strain * Genotype + Sex)  2719.898065    2.882356
8                        m8 (Strain * Sex + Genotype)  2718.776604    1.760895
9                        m9 (Strain * Sex * Genotype)  2718.5

In [16]:
model_data = rest_transformed_df.copy()

# ============================================================
# Add Background column
# ============================================================

model_data["Background"] = np.where(
    model_data["Strain"] == "B6",
    "Same",
    "Mixed"
)

# Optional: make ordering explicit
model_data["Background"] = pd.Categorical(
    model_data["Background"],
    categories=["Same", "Mixed"]
)

print(model_data[["Strain", "Background"]].value_counts())


# ============================================================
# Original OLS models using Strain
# ============================================================

m0 = smf.ols(
    "immobile_time ~ 1",
    data=model_data
).fit()

m1 = smf.ols(
    "immobile_time ~ Strain",
    data=model_data
).fit()

m2 = smf.ols(
    "immobile_time ~ Sex",
    data=model_data
).fit()

m3 = smf.ols(
    "immobile_time ~ Strain + Sex",
    data=model_data
).fit()

m4 = smf.ols(
    "immobile_time ~ Strain * Sex",
    data=model_data
).fit()

m5 = smf.ols(
    "immobile_time ~ Strain + Sex + Genotype",
    data=model_data
).fit()

m6 = smf.ols(
    "immobile_time ~ Strain + Sex * Genotype",
    data=model_data
).fit()

m7 = smf.ols(
    "immobile_time ~ Strain * Genotype + Sex",
    data=model_data
).fit()

m8 = smf.ols(
    "immobile_time ~ Strain * Sex + Genotype",
    data=model_data
).fit()

m9 = smf.ols(
    "immobile_time ~ Strain * Sex * Genotype",
    data=model_data
).fit()


# ============================================================
# Parallel models using Background instead of Strain
# ============================================================

# Equivalent of m1
m10 = smf.ols(
    "immobile_time ~ Strain +  Sex + Genotype + Background:Genotype",
    data=model_data
).fit()

# Equivalent of m3
m11 = smf.ols(
    "immobile_time ~ Strain + Sex + Genotype + Strain:Sex + Sex:Genotype + Background:Genotype + Sex:Background:Genotype",
    data=model_data
).fit()

# ============================================================
# AIC comparison
# ============================================================

aic_values = {

    # Shared models
    "m0 (null)": m0.aic,
    "m2 (Sex)": m2.aic,

    # Strain models
    "m1 (Strain)": m1.aic,
    "m3 (Strain + Sex)": m3.aic,
    "m4 (Strain * Sex)": m4.aic,
    "m5 (Strain + Sex + Genotype)": m5.aic,
    "m6 (Strain + Sex * Genotype)": m6.aic,
    "m7 (Strain * Genotype + Sex)": m7.aic,
    "m8 (Strain * Sex + Genotype)": m8.aic,
    "m9 (Strain * Sex * Genotype)": m9.aic,

    # Background models
    "m10 (Strain + Sex + Genotype + SameBackground:Genotype)": m10.aic,
    "m11 (Strain + Sex + Genotype + Strain:Sex + Sex:Genotype + SameBackground:Genotype + Sex:SameBackground:Genotype)": m11.aic
}


# ============================================================
# Make AIC dataframe
# ============================================================

aic_df = pd.DataFrame({
    "Model": list(aic_values.keys()),
    "AIC": list(aic_values.values())
})

aic_df["ΔAIC"] = aic_df["AIC"] - aic_df["AIC"].min()



print(aic_df)
aic_df.to_excel(RESULTS_DIR / 'AIC_table_FST_4mins_Sex_Strain_Genotype_newmethod_fighting.xlsx', index=False)

Strain  Background
B6      Same          106
DBA     Mixed          99
CBA     Mixed          97
MRL     Mixed          93
Name: count, dtype: int64
                                                Model           AIC  \
0                                           m0 (null)  17536.937604   
1                                            m2 (Sex)  17180.748583   
2                                         m1 (Strain)  17232.199174   
3                                   m3 (Strain + Sex)  16883.071647   
4                                   m4 (Strain * Sex)  16887.066884   
5                        m5 (Strain + Sex + Genotype)  16885.071643   
6                        m6 (Strain + Sex * Genotype)  16885.846356   
7                        m7 (Strain * Genotype + Sex)  16890.355957   
8                        m8 (Strain * Sex + Genotype)  16889.063063   
9                        m9 (Strain * Sex * Genotype)  16894.606840   
10  m10 (Strain + Sex + Genotype + SameBackground:...  16886.707299   